## Upload data to local Postgre SQL database

Cálculo de la ampacidad IEEE y CIGRE con la implementación UC. El script está diseñado para trabajar con los ficheros .csv de GE descargados de Internet, así como de Ampacimon descargados de la web de Iberdrola. 

Se utilizan los ficheros  LLLLL-AAAA-MM-DD-BBBB-NN-EE-XXXXX-SpanRating

LLLLL .- línea

AAAA .- año de inicio
MM .- mes de inicio
DD .- día de inicio

BBBB .- año de fin
NN .- mes de fin
EE .- día de fin

XXXXX .- Apoyo   

Las columnas del fichero son:

lineId;
spanId;
ampacimonId;
measurementTime [UTC];
measurementTime [Local];
computationTime [UTC];
computationTime [Local];
sag [m];
temperatureFromSag [Celsius degree];
smoothedAmpacity [Ampere];
smoothedAmpacity [MVA];
ampacityAmbientTemperature [Ampere];
ampacityAmbientTemperature [MVA];
lineCurrent [Ampere];
lineCurrent [MVA];
staticRating [Ampere];
staticRating [MVA];
initialPerpendicularWindSpeed [Meter per second];
initialAmbientTemperature [Celsius degree];
initialSolarRadiation [Watt per square meter];
ampacity5min [Ampere];
ampacity5min [MVA];
ampacity10min [Ampere];
ampacity10min [MVA];
ampacity15min [Ampere];
ampacity15min [MVA];
ampacity20min [Ampere];
ampacity20min [MVA];
ampacity30min [Ampere];
ampacity30min [MVA];
ampacity45min [Ampere];
ampacity45min [MVA];
sagAlarm;conductorTemperatureAlarm;
lineCurrentAlarm;
iceAlarm;
quality;
qualityReason;
iceWeigth;
lineVoltage;
clearanceAtGivenLocation;
rotationAngle;
rotationAngleChange;
iceRiskValue;
rainfall;
cumulativeRainfallDuringFreezing


Ordenarlos en carpetas atendiendo al nombre de la línea y definir los arrays y variable siguientes antes de ejecutar:   

patha = ['e:/GE_Internet/ElPalmar_Espinardo/', 'e:/GE_Internet/Hellin_Calasparra/']   
LineNamea = ['EL PALMAR - ESPINARDO', 'HELLÍN - CALASPARRA']   
table_name    


**author**: GTEA-UC  
**email**: mananam@unican.es  
**last update**: 04/02/2024

In [ ]:
import sys 
print( sys.version)

In [ ]:
import psycopg2
import pyodbc
import configparser # Config file
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from datetime import datetime, timedelta
import calendar
import time
from dateutil.relativedelta import relativedelta
from sqlalchemy import create_engine
import os

In [ ]:
# Función para convertir cada string a datetime a partir del formato del fichero original
# '%Y-%m-%dT%H:%M:%S.%fZ'
# a
# '%Y-%m-%d %H:%M:%S'
#
def convertir_a_datetime(cadena):
    # Convertir la cadena a un objeto datetime
    fecha_obj = datetime.strptime( cadena, '%Y-%m-%dT%H:%M:%S.%fZ')
    # Formatear el objeto datetime al nuevo formato '%Y-%m-%d %H:%M:%S'
    fecha_nuevo_formato = fecha_obj.strftime('%Y-%m-%d %H:%M:%S')
     
    # si necesitas datetime
    # datetime.strptime( fecha_nuevo_formato, '%Y-%m-%d %H:%M:%S')  
     
    return   fecha_nuevo_formato
    #if isinstance( cadena, str):
    #    return datetime.strptime(cadena, '%Y-%m-%dT %H:%M:%S')
    #else:
    #    return cadena

In [ ]:
# Ruta a la carpeta de ficheros descargados de la web de Iberdrola de Ampacimon
ampacimon_path = 'E:\\Ampacimon_Iberdrola\\'

# Ficheros a cargar
ampacimon_files = ['IBER_ROCA_CARRUS-2023-01-01-2023-12-31-10155-SpanRating.csv',
         'IBER_ROCA_CARRUS-2023-01-01-2023-12-31-10156-SpanRating.csv']

# Nombre de la línea a la que hace referencia cada fichero
ampacimon_lines = ['ROCAMORA - CARRUS', 'ROCAMORA - CARRUS']

# Nombre del apoyo en el que se ubica cada fichero
ampacimon_spam = ['10073-10074', '10136-10137']

In [ ]:
# Obtiene el número de ficheros a cargar
Nlines = len(ampacimon_files)

columnas_seleccionadas = ['TimeStamp', 'LineName', 'NodeName', 'Ampacimon']
df_A = pd.DataFrame(columns=columnas_seleccionadas)

# Para todos los ficheros
for index,value in enumerate(ampacimon_files):
    full_path = ampacimon_path + value
    print( full_path)
    df = pd.read_csv( full_path, sep=';')

    # Ahora actualizando correctamente solo algunos nombres de columnas
    df.rename(columns={'lineId':'LineName',
                   'spanId':'NodeName',
                   'measurementTime [UTC]':'TimeStamp',
                   'lineCurrent [Ampere]':'PhaseCurrent',
                   'temperatureFromSag [Celsius degree]':'ConductorTemp', 
                   'initialAmbientTemperature [Celsius degree]':'AmbientTemp',
                   'initialPerpendicularWindSpeed [Meter per second]':'WindSpeed',
                   'initialSolarRadiation [Watt per square meter]':'SolarRadiation',
                   'smoothedAmpacity [Ampere]':'Ampacimon'}, inplace=True)
    
    
    df['TimeStamp'] = df['TimeStamp'].apply(convertir_a_datetime)
    print('Tamaño: ' + str(len(df)))
    
    
    df_nuevo = df[columnas_seleccionadas]
    
    df_A = pd.concat([df_A, df_nuevo], ignore_index=True)

 
df_A['LineName'] = 'ROCAMORA - CARRUS'   
print('Tamaño: ' + str(len(df_A)))
    

Valor de fecha y hora a buscar  

search_value = datetime(2023, 2, 27, 12, 24, 0)

Obteniendo el índice del DataFrame donde la fila de fecha y hora es igual al valor buscado  

index_result = df.index[ df['TimeStamp'] == search_value].tolist()  

index_result

In [ ]:
Nlines

    column_names = [
            'LineName',
            'NodeName',
            'TimeStamp',
            'PhaseCurrent',
            'PhasePhase' ,
            'ConductorTemp' ,
            'AmbientTemp' ,
            'WindSpeed' ,
            'WindDomDirection' ,
            'WindAvgDirection' ,
            'SolarRadiation' ,
            'DewPoint' ,
            'ServiceName' ,
            'Clearance' ,
            'IMAX' , 
            'LoadMVA' ,
            'MaxCapacityMVA' ,
            'IEEE738' ,
            'CIGRE601' ,
            'ucIEEE738' ,
            'ucCIGRE601' ,
            'ucCIGRE207']

**Paquetes específicos DLR**

In [ ]:
from cable import cable
from case import case
from ieee738 import ieee738
from cigre601 import cigre601
from cigre207 import cigre207
from pvsystems import pvsystems
import matplotlib.pyplot as plt 


# Needed only during the development phase.
from importlib import reload
reload( cable)
reload( case)
reload( ieee738)
reload( cigre601)
reload( cigre207)
reload( pvsystems)

Upload the dataframe to the database table

Information for the connection to the database

In [ ]:
# PostgreSQL connection parameters
db_params = {
    'host': 'localhost', # the database is in the same machine where this script is running
    'database': 'Iberdrola', # name of the database
    'user': 'postgres', # user
    'password': 'mananam05' # password
}

Create a connection to the database. 

Define the name of the table where the transactions will take place.

In [ ]:
# Establish a connection to the PostgreSQL database using SQLAlchemy
connection_string = f"postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params['host']}/{db_params['database']}"
engine = create_engine(connection_string)

# Define the name of the table you want to create or replace
table_name = 'ucgeproc33'

patha .- is the path to the folder where the information about the lines is storaged.

LineName .- is the array with the names of the lines. This array is defined according patha


In [ ]:
patha = ['e:/GE_Internet/ElPalmar_Espinardo/', 'e:/GE_Internet/Hellin_Calasparra/']   
LineNamea = ['EL PALMAR - ESPINARDO', 'HELLÍN - CALASPARRA']  

#patha = ['e:/GE3/HC/', 'e:/GE3/PE/']
#LineNamea = ['Hellín-Calasparra', 'El Palmar-Espinardo']

#patha = [ 'e:/GE44/']
#LineNamea = [ 'HELLÍN - CALASPARRA']

file_case = "E:/mario/trabajos2/iberdrola_DTR/datos/case.xlsx" # Information about the cases
file_cable = "E:/mario/trabajos2/iberdrola_DTR/datos/cable.xlsx" # Information about the cables


case_dtypes={ 'LineName': str,
             'NodeName': str}


cable_dtypes={ 'LineName': str}

#dfCases = pd.read_excel( file_case, header=0)
#dfCables = pd.read_excel( file_cable, header=0)

dfCases = pd.read_excel( file_case, header=0, dtype=case_dtypes)
dfCases['LineName'] = dfCases['LineName'].astype( str)
dfCases['NodeName'] = dfCases['NodeName'].astype( str)
dfCables = pd.read_excel( file_cable, header=0, dtype=cable_dtypes)
dfCables['LineName'] = dfCables['LineName'].astype( str)

In [ ]:
dfCases

In [ ]:
dfCables

In [ ]:
def get_df( data_frame, LineName, NodeName, dfCables, dfCases, table_name, engine, df_A):
    
    column_names = [
            'LineName',
            'NodeName',
            'TimeStamp',
            'PhaseCurrent',
            'PhasePhase' ,
            'ConductorTemp' ,
            'AmbientTemp' ,
            'WindSpeed' ,
            'WindDomDirection' ,
            'WindAvgDirection' ,
            'SolarRadiation' ,
            'DewPoint' ,
            'ServiceName' ,
            'Clearance' ,
            'IMAX' , 
            'LoadMVA' ,
            'MaxCapacityMVA' ,
            'geIEEE' ,
            'geCIGRE' ,
            'ucIEEE738' ,
            'ucCIGRE601' ,
            'ucCIGRE207']
    
    
    TotalErrors = 0
    TotalRows = len( data_frame) #Tamaño del array a procesar
    TotalCount = 1 # Contador de 1 a TotalRows
    PrintEach = 1000 # Imprime cada [PrientEach] iteracciones
    PrintControl = 1 # Contador para activar una vez cada [PrintEach]    
    
    df2 = pd.DataFrame( columns = column_names) # define a new dataframe with the columns defined by the array column_names.
    for index, row in data_frame.iterrows():
        
        #LineName1 = str(LineName)
        #NodeName1 = str(NodeName)
        datetime_str = str(row['Date']) + "  " + str(row['Time'])
        datetime_obj = datetime.strptime( datetime_str, "%d/%m/%Y %H:%M")
        TimeStamp = datetime_obj.strftime('%Y-%m-%d %H:%M:%S') 
        #print('\r' + TimeStamp, end=' ', flush=True)

        dfCable = dfCables[ dfCables['LineName']==LineName]
        #print("***************************************")
        #print('LineName: ' + LineName)
        #print('NodeName: ' + NodeName)
        #print(dfCable)
        dfCase = dfCases[ dfCases['NodeName']==NodeName]
        #print("-----------------------------------------")
        #print(dfCase)
        #print("***************************************")

        AmbientTemp = float( row['Temperature'])
        #Temperature = float(row['Temperature'])
        WindspeedAVG = float(row['Windspeed AVG'])
        WindDirectionAVG = float(row['Wind Direction AVG'])
        WindDirectionDominat = float(row['Wind Direction Dominant'])
        SolarRadiation = float(row['Solar Radiation'])
        DewPointTemperature = float(row['Dew Point Temperature'])
        WindSonic = float(row['WindSonic Windspeed AVG'])
        WindSonicWindDirectionAVG = float(row['WindSonic Wind Direction AVG'])
        WindSonicWindDirectionDominant = float(row['WindSonic Wind Direction Dominant'])
        PhaseCurrent = float(row['I'])
        ConductorTemp = float(row['T'])
        RCCIEEE = float(row['RCC IEEE'])
        RCCCIGRE = float(row['RCC CIGRE'])
        Clearance = float(row['CLEARANCE'])
        RIME = float(row['RIME'])
    

        NSELECT = 2 
        Cable1 = cable.Cable()
        #c_db, error = Cable1.load_cable_db()
        #Cable1.set_cable( NSELECT, conductor = dfCable.at[1,'Cstring'])

        Cable1.D = 1000*float( dfCable['D'])
        #Cable1.C = 10.4
        Cable1.d = 1000*float( dfCable['d']) 
        Cable1.TLO = float( dfCable['TLO']) 
        Cable1.THI = float( dfCable['THI']) 
        #Cable1.TCDRMAX = 60.0
        Cable1.RLO = float( dfCable['RLO']) 
        Cable1.RHI = float( dfCable['RHI']) 

        Cable1.EMISS = float( dfCable['EMISS'])
        Cable1.ABSORP = float( dfCable['ABSORP'])


        Cable1.HNH = int( dfCable['HNH']) 
        #Cable1.HEATOUT = 357.9
        #Cable1.HEATCORE = 132.1
        Cable1.TotalS = float( dfCable[ 'TotalS']) 
        Cable1.CSteel20 = float( dfCable['CSteel20']) 
        Cable1.CAlum20 = float( dfCable[ 'CAlum20']) 
        Cable1.BetaSteel20 = float( dfCable[ 'BetaSteel20']) 
        Cable1.BetaAlum20 = float( dfCable[ 'BetaAlum20']) 
        Cable1.mSteel = float( dfCable[ 'mSteel']) 
        Cable1.mAlum = float( dfCable[ 'mAlum']) 


        Case1 = case.Case()
        Case1.demo( NSELECT)
        # Ambient conditions
        Case1.TAMB = AmbientTemp
        try:
            Case1.CDR_LAT_DEG = float( dfCase['CDR_LAT_DEG'])
        except Exception as e:
            print(NodeName)
            break        
        
        Case1.ALBEDO = float(dfCase['ALBEDO'])
        Case1.beta = 0
        Case1.CDR_ELEV = float(dfCase['CDR_ELEV'])
        Case1.TCDRPRELOAD = float( dfCase['TCDR'])
        #Case1.TCDRMAX = 150
        #Case1.TCDR = 100.0
        Case1.SolarRadiation = SolarRadiation
        if WindspeedAVG < 0.1:
            WindspeedAVG += 0.01        
        Case1.VWIND = WindspeedAVG 
        Case1.WINDANG_DEG = abs(WindDirectionAVG-float( dfCase['ANG_DEG']))
        Case1.Z1_DEG = float(dfCase['Z1_DEG'])
        Case1.Ns = 1.0
        Case1.SUN_TIME = 99 # solar Radiation measured (IEEE738)
        Case1.SOLAR = 0 # Solar Radiation measured (CIGRE601)
        dia = int(datetime_obj.strftime('%d'))
        mes = int(datetime_obj.strftime('%m') )
        #Case1.NDAY = PV1.DayOfYear( dia, mes) # 10th June
        #print("NDAY: " + str(Case1.NDAY))

        # IEEE 738
        X1 = ieee738.IEEE738()
        X1.Debug = 0
        X1.set_cable( Cable1)
        X1.set_case( Case1)
        X1.Case1.SORM = 1
        X1.ieee_738_2013()      
        #X1.output()
        ucIEEE738 = round( float(X1.Case1.TR), 2)

        # CIGRE TB 601
        X2 = cigre601.CIGRE601()
        X2.Debug = 0
        X2.set_error( 0) # no error
        X2.set_cable( Cable1)
        X2.set_case( Case1)
        X2.cigre601()    
        #X2.output()
        ucCIGRE601 =round( float(X2.Case1.TR), 2)


        # CIGRE TB 207
        X3 = cigre207.CIGRE207()
        X3.Debug = 0
        X3.set_error( 0) # no error
        X3.set_cable( Cable1)
        X3.set_case( Case1)
        X3.cigre207()    
        #X3.output()
        ucCIGRE207 =round( float(X3.Case1.TR), 2)
        
    
    
        new_row_data = {
            'LineName' : LineName,
            'NodeName' : NodeName,
            'TimeStamp' : TimeStamp,
            'PhaseCurrent' : PhaseCurrent,
            'PhasePhase' : ' ',
            'ConductorTemp' : ConductorTemp,
            'AmbientTemp' : AmbientTemp,
            'WindSpeed' : WindspeedAVG,
            'WindDomDirection' : WindDirectionDominat,
            'WindAvgDirection' : WindDirectionAVG,
            'SolarRadiation' : SolarRadiation,
            'DewPoint' : DewPointTemperature,
            'ServiceName' : ' ',
            'Clearance' : Clearance,
            'IMAX' : ' ',
            'LoadMVA' : ' ',
            'MaxCapacityMVA' : ' ',
            'geIEEE' : RCCIEEE,
            'geCIGRE' : RCCCIGRE,
            'ucIEEE738' : ucIEEE738,
            'ucCIGRE601' : ucCIGRE601,
            'ucCIGRE207': ucCIGRE207}


        new_row_type = {
            'LineName' : str,
            'NodeName' : str,
            'TimeStamp' : str,
            'PhaseCurrent' : float, 
            'PhasePhase' : float, 
            'ConductorTemp' : float, 
            'AmbientTemp' : float, 
            'WindSpeed' : float, 
            'WindDomDirection' : float, 
            'WindAvgDirection' : float, 
            'SolarRadiation' : float, 
            'DewPoint' : float, 
            'ServiceName' : str,
            'Clearance' : Clearance,
            'IMAX' : float,
            'LoadMVA' : float, 
            'MaxCapacityMVA' :  float, 
            'geIEEE' : float, 
            'geCIGRE' : float, 
            'ucIEEE738' : float, 
            'ucCIGRE601' : float,
            'ucCIGRE207' : float}


        CIGRE601error = X2.get_error()
        CIGRE207error = X3.get_error()
        if (CIGRE601error == 0) and (CIGRE207error == 0):
            new_df = pd.DataFrame( [new_row_data]) #, dtype=new_row_data)
            df2 = pd.concat( [df2, new_df], ignore_index=True)
        else:
            TotalErrors += 1
        
        
        TotalCount += 1
        PrintControl += 1


        # Fusiona dataframes
        df3 = pd.merge( df2, df_A, on=['TimeStamp', 'LineName', 'NodeName'], how='left')


        # Cada PrintEach imprime situación y almacena en la BBDD
        if PrintControl == PrintEach:
            PrintControl = 1
            if TotalRows > 0:
                PercTotal = round(100*TotalCount/TotalRows,1) 
                print('\r' + TimeStamp + '; ' + LineName + '; ' + NodeName + '; Percentage: ' + str(PercTotal) + ' %                                 ', end=' ', flush=True)    
                df3.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
            else:
                print('\r No data in this file...')
            df2 = pd.DataFrame( columns = column_names)


    # almacena en la BBDD lo que queda
    df3.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
    if TotalRows > 0:
        PercTotal = round(100*TotalCount/TotalRows,2) 
        print('\r' + TimeStamp + '; ' + LineName + '; ' + NodeName + '; Percentage: ' + str(PercTotal) + ' %  ; Total Count [rows]: ' + str(TotalCount) + '                                    ', end=' ', flush=True)    
    else:
        print('\r No data in this file...')  
     
    return( TotalErrors)
    

In [ ]:
dtype_mapping = {
            'LineName' : 'text',
            'NodeName' : 'text',
            'TimeStamp' : 'timestamp',
            'PhaseCurrent' : 'double precision',
            'PhasePhase' : 'double precision',
            'ConductorTemp' : 'double precision',
            'AmbientTemp' : 'double precision',
            'WindSpeed' : 'double precision',
            'WindDomDirection' : 'double precision',
            'WindAvgDirection' : 'double precision',
            'SolarRadiation' : 'double precision',
            'DewPoint' : 'double precision',
            'ServiceName' : 'text',
            'Clearance' : 'double precision',
            'IMAX' : 'double precision',
            'LoadMVA' : 'double precision',
            'MaxCapacityMVA' : 'double precision',
            'IEEE738' : 'double precision',
            'CIGRE601' : 'double precision',
            'ucIEEE738' : 'double precision',
            'ucCIGRE601' : 'double precision'}


Errors = 0 # Total number of errors and inconsistencies

for index_p, value_p in enumerate( patha):
    file_list = os.listdir( patha[index_p])
    print( file_list)
    PV1 = pvsystems.PVSystems()
    
    for index, value in enumerate( file_list):
        LineName = LineNamea[ index_p]
        split_list = value.split(" ")
        NodeName = split_list[1] # ie.['Apoyo', '10174', '2023.csv']
        
        #print( NodeName)
        Year_list = split_list[2].split(".") 
        Year = Year_list[0]
        #print( YearMonth)
        #y1 = YearMonth[0]
        #y2 = YearMonth[1]
        #yy = y1 + y2
        #year = 2000 + int( yy)
        #print(year)
        #m1 = YearMonth[2]
        #m2 = YearMonth[3]
        #mm = m1 + m2
        #month = int(mm)
        fullpath = patha[index_p] + file_list[index]
        print( "%s ### %s - %s. %s" %(fullpath, LineName, NodeName,  Year))
        inicio = time.time()
        
        data_frame2 = pd.read_csv( fullpath, delimiter=';', header=0, decimal=',')
        
        # Filter colums with missing values
        columns_to_check = [
            'Date', 
            'Time', 
            'Temperature', 
            'Windspeed AVG', 
            'Wind Direction AVG',
            'Wind Direction Dominant', 
            'Solar Radiation', 
            'Dew Point Temperature',
            'I', 
            'T' 
            ]
        data_frame3 = data_frame2.dropna( subset=columns_to_check)
        #data_frame4 = data_frame3[ data_frame3['ServiceName'] == 'RCC CIGRE']
        #data_frame4['IMAX'] = pd.to_numeric( data_frame4['IMAX'], errors='coerce')
        #data_frame5 = data_frame4[ data_frame4['IMAX'] > 0.0]        
        
        ierrors = get_df( data_frame3, LineName, NodeName, dfCables, dfCases, table_name, engine, df_A)
        Errors += ierrors
        #print(df22)
        #df22.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid', dtype=dtype_mapping) 
        #df22.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
        
        fin = time.time()
        tiempo_total = round((fin - inicio)/60.0,1)
        print("                                                                   ")
        print(f'Execution time: {tiempo_total} minutos')
        print('Inconsistencies: %i' %(ierrors))
        print("********************************************************************************")
        print("********************************************************************************")


print("Total number of inconsistencies: %i" %(Errors))            
        
        

In [ ]:
df2

In [ ]:
# Close the database connection
engine.dispose()